In [1]:
!pip install segmentation-models-pytorch albumentations -q

import os
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"ahnafnafim","key":"YOUR_KEY"}')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle datasets download -d araftahsanpavel/tgif-subset -p /kaggle/working/

import zipfile
with zipfile.ZipFile('/kaggle/working/tgif-subset.zip', 'r') as z:
    z.extractall('/kaggle/working/')

# Free space immediately
os.remove('/kaggle/working/tgif-subset.zip')
print("✓ Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/araftahsanpavel/tgif-subset
License(s): unknown
100%|███████████████████████████████████████| 8.88G/8.88G [01:15<00:00, 127MB/s]

✓ Done


In [2]:
import os, time, random, json
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# ── Hyperparameters — IDENTICAL to ForensicNet/SegFormer ──
IMAGE_SIZE          = 384
EPOCHS              = 20
BATCH_SIZE          = 8
LEARNING_RATE       = 1e-4
WEIGHT_DECAY        = 1e-4
LR_PATIENCE         = 3
LR_FACTOR           = 0.5
LR_MIN              = 1e-6
EARLY_STOP_PATIENCE = 7
BINARY_THRESHOLD    = 0.5
GRAD_CLIP_NORM      = 1.0
MIXED_PRECISION     = True
SEED                = 42
NUM_WORKERS         = 4
PIN_MEMORY          = True
POS_WEIGHT          = 3.0

# Paths
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = MIXED_PRECISION and torch.cuda.is_available()

DATA_ROOT = "/kaggle/working/subset"
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

# For compatibility with your existing cells
IMG_SIZE = IMAGE_SIZE
NUM_EPOCHS = EPOCHS
LR = LEARNING_RATE
GRAD_CLIP = GRAD_CLIP_NORM
PATIENCE = EARLY_STOP_PATIENCE

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    print(f"Device: {DEVICE} — {torch.cuda.get_device_name(0)}")
else:
    print(f"Device: {DEVICE}")

Device: cuda — Tesla T4


In [3]:
class TGIFSubsetDataset(Dataset):
    def __init__(self, root, split='training', transform=None, include_authentic=True):
        self.transform = transform
        self.samples = []

        split_dir = os.path.join(root, split)
        fakes_dir = os.path.join(split_dir, 'fakes')
        masks_dir = os.path.join(split_dir, 'masks')
        images_dir = os.path.join(split_dir, 'images')

        categories = []
        if os.path.isdir(fakes_dir):
            categories = [d for d in os.listdir(fakes_dir)
                         if os.path.isdir(os.path.join(fakes_dir, d))]

        for cat in sorted(categories):
            cat_fakes = os.path.join(fakes_dir, cat)
            cat_masks = os.path.join(masks_dir, cat)
            cat_images = os.path.join(images_dir, cat)

            mask_files = {}
            if os.path.isdir(cat_masks):
                for f in os.listdir(cat_masks):
                    if os.path.splitext(f)[1].lower() in IMG_EXTS:
                        mask_files[f] = os.path.join(cat_masks, f)

            if os.path.isdir(cat_fakes):
                for fake_name in sorted(os.listdir(cat_fakes)):
                    if os.path.splitext(fake_name)[1].lower() not in IMG_EXTS:
                        continue
                    for mask_name, mask_path in mask_files.items():
                        if fake_name.startswith(mask_name):
                            self.samples.append((os.path.join(cat_fakes, fake_name), mask_path, 1))
                            break

            if include_authentic and os.path.isdir(cat_images):
                for f in sorted(os.listdir(cat_images)):
                    if os.path.splitext(f)[1].lower() in IMG_EXTS:
                        self.samples.append((os.path.join(cat_images, f), None, 0))

        n_forged = sum(1 for s in self.samples if s[2] == 1)
        n_auth = sum(1 for s in self.samples if s[2] == 0)
        print(f"  {split}: {len(self.samples)} total ({n_forged} forged, {n_auth} authentic)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path, label = self.samples[idx]
        image = np.array(Image.open(img_path).convert('RGB'))
        if mask_path is not None:
            mask_img = Image.open(mask_path).convert('L')
            if mask_img.size != (image.shape[1], image.shape[0]):
                mask_img = mask_img.resize((image.shape[1], image.shape[0]), resample=Image.NEAREST)
            mask = np.array(mask_img)
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.float32)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask  = augmented['mask']
        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)
        mask  = mask.unsqueeze(0).float()
        label = torch.tensor(float(label))
        return image, mask, label


# No augmentation — baseline
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

print("Building datasets (no augmentation)...")
train_ds = TGIFSubsetDataset(DATA_ROOT, 'training', transform=train_tf)
val_ds   = TGIFSubsetDataset(DATA_ROOT, 'validation', transform=val_tf)
test_ds  = TGIFSubsetDataset(DATA_ROOT, 'testing', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

batch = next(iter(train_loader))
print(f"\nBatch — Image: {batch[0].shape}, Mask: {batch[1].shape}, Label: {batch[2].shape}")

Building datasets (no augmentation)...
  training: 7559 total (5459 forged, 2100 authentic)
  validation: 1023 total (682 forged, 341 authentic)
  testing: 1029 total (686 forged, 343 authentic)

Batch — Image: torch.Size([8, 3, 384, 384]), Mask: torch.Size([8, 1, 384, 384]), Label: torch.Size([8])


In [4]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0, pos_weight=3.0):
        super().__init__()
        self.smooth = smooth
        self.register_buffer(
            "pos_weight",
            torch.tensor([pos_weight], dtype=torch.float32)
        )

    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=self.pos_weight.to(logits.device)
        )

        preds = torch.sigmoid(logits).view(logits.size(0), -1)
        tgt = targets.view(targets.size(0), -1)

        intersection = (preds * tgt).sum(dim=1)
        union = preds.sum(dim=1) + tgt.sum(dim=1)

        dice_loss = 1.0 - ((2.0 * intersection + self.smooth) / (union + self.smooth)).mean()

        return 0.5 * bce_loss + 0.5 * dice_loss


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    start = time.time()

    for images, masks, labels in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)

        preds = (torch.sigmoid(logits) > BINARY_THRESHOLD).float()
        correct += (preds == masks).sum().item()
        total += masks.numel()

    return total_loss / len(loader.dataset), correct / total, time.time() - start


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    for images, masks, labels in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with autocast("cuda", enabled=USE_AMP):
            logits = model(images)
            loss = criterion(logits, masks)

        total_loss += loss.item() * images.size(0)

        preds = (torch.sigmoid(logits) > BINARY_THRESHOLD).float()
        correct += (preds == masks).sum().item()
        total += masks.numel()

    return total_loss / len(loader.dataset), correct / total


@torch.no_grad()
def full_evaluation(model, loader, device, threshold=0.5):
    model.eval()
    all_pp, all_pt, all_ip, all_it = [], [], [], []

    for images, masks, labels in loader:
        images = images.to(device, non_blocking=True)

        with autocast("cuda", enabled=USE_AMP):
            logits = model(images)

        probs = torch.sigmoid(logits).cpu().numpy()
        masks_np = masks.numpy()
        labels_np = labels.numpy()

        for i in range(probs.shape[0]):
            pred = (probs[i, 0] > threshold).astype(np.int32).flatten()
            gt = masks_np[i, 0].astype(np.int32).flatten()

            all_pp.append(pred)
            all_pt.append(gt)

            all_ip.append(1 if pred.max() > 0 else 0)
            all_it.append(int(labels_np[i]))

    pp, pt = np.concatenate(all_pp), np.concatenate(all_pt)
    ip, it = np.array(all_ip), np.array(all_it)

    intersection = ((pp == 1) & (pt == 1)).sum()
    union = ((pp == 1) | (pt == 1)).sum()

    return {
        "pixel": {
            "accuracy": accuracy_score(pt, pp),
            "precision": precision_score(pt, pp, zero_division=0),
            "recall": recall_score(pt, pp, zero_division=0),
            "f1": f1_score(pt, pp, zero_division=0),
            "iou": float(intersection / (union + 1e-8)),
            "dice": float(2 * intersection / (pp.sum() + pt.sum() + 1e-8)),
        },
        "image": {
            "accuracy": accuracy_score(it, ip),
            "precision": precision_score(it, ip, zero_division=0),
            "recall": recall_score(it, ip, zero_division=0),
            "f1": f1_score(it, ip, zero_division=0),
        },
        "pixel_cm": confusion_matrix(pt, pp, labels=[0, 1]),
        "image_cm": confusion_matrix(it, ip, labels=[0, 1]) if len(set(it)) > 1 else np.array([[0]]),
    }

print("✓ Functions ready")

✓ Functions ready


In [8]:
model_name = "PVTv2B2_384_noaug_posweight"

print(f"{'='*70}")
print(f"TRAINING: {model_name}")
print("Config: 384x384, no augmentation, pos_weight=3.0")
print(f"{'='*70}")

# Recommended clean transformer baseline
# If this works and you want a stronger/heavier model later, try:
# encoder_name = "tu-pvt_v2_b3"
encoder_name = "tu-pvt_v2_b2"

model = smp.Unet(
    encoder_name=encoder_name,
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_size = total_params * 4 / 1024**2

print(f"Encoder: {encoder_name}")
print(f"Parameters: {total_params:,}")
print(f"Trainable: {trainable_params:,}")
print(f"Size: {model_size:.1f} MB")

criterion = DiceBCELoss(pos_weight=POS_WEIGHT).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=LR_MIN
)

scaler = GradScaler("cuda", enabled=USE_AMP)

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "val_f1": [],
    "val_iou": [],
    "val_precision": [],
    "val_recall": [],
    "epoch_time": [],
    "lr": []
}

best_val_iou = -1.0
best_val_f1 = -1.0
best_epoch = 0
total_time = 0
patience_counter = 0

best_ckpt_path = f"{SAVE_DIR}/best_{model_name}.pth"
latest_ckpt_path = f"{SAVE_DIR}/latest_{model_name}.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    tl, ta, et = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        DEVICE
    )

    vl, va = validate(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    val_eval = full_evaluation(
        model,
        val_loader,
        DEVICE,
        threshold=BINARY_THRESHOLD
    )

    val_iou = val_eval["pixel"]["iou"]
    val_f1 = val_eval["pixel"]["f1"]
    val_precision = val_eval["pixel"]["precision"]
    val_recall = val_eval["pixel"]["recall"]

    scheduler.step(val_iou)

    current_lr = optimizer.param_groups[0]["lr"]
    total_time += et

    history["train_loss"].append(tl)
    history["val_loss"].append(vl)
    history["train_acc"].append(ta)
    history["val_acc"].append(va)
    history["val_f1"].append(val_f1)
    history["val_iou"].append(val_iou)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)
    history["epoch_time"].append(et)
    history["lr"].append(current_lr)

    # Save latest every epoch
    torch.save({
        "model_state_dict": model.state_dict(),
        "epoch": epoch,
        "history": history,
        "best_val_iou": best_val_iou,
        "best_val_f1": best_val_f1,
        "config": {
            "model_name": model_name,
            "encoder_name": encoder_name,
            "image_size": IMAGE_SIZE,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "pos_weight": POS_WEIGHT,
            "augmentation": False,
            "binary_threshold": BINARY_THRESHOLD,
            "grad_clip_norm": GRAD_CLIP_NORM
        }
    }, latest_ckpt_path)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_val_f1 = val_f1
        best_epoch = epoch
        patience_counter = 0

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "best_val_iou": best_val_iou,
            "best_val_f1": best_val_f1,
            "history": history,
            "total_params": total_params,
            "trainable_params": trainable_params,
            "model_size_mb": model_size,
            "config": {
                "model_name": model_name,
                "encoder_name": encoder_name,
                "image_size": IMAGE_SIZE,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "pos_weight": POS_WEIGHT,
                "augmentation": False,
                "binary_threshold": BINARY_THRESHOLD,
                "grad_clip_norm": GRAD_CLIP_NORM
            }
        }, best_ckpt_path)

        improved = "YES"
    else:
        patience_counter += 1
        improved = "NO"

    print(f"Epoch {epoch:>3}/{NUM_EPOCHS} | "
          f"TrLoss: {tl:.4f} TrAcc: {ta:.4f} | "
          f"VlLoss: {vl:.4f} VlAcc: {va:.4f} | "
          f"ValF1: {val_f1:.4f} ValIoU: {val_iou:.4f} | "
          f"LR: {current_lr:.2e} | "
          f"{et:.1f}s | Improved: {improved} | "
          f"Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered after {PATIENCE} epochs without improvement.")
        break

history["total_time"] = total_time
history["best_epoch"] = best_epoch
history["best_val_iou"] = best_val_iou
history["best_val_f1"] = best_val_f1

print("\nTraining complete.")
print(f"Best epoch: {best_epoch}")
print(f"Best val IoU: {best_val_iou:.4f}")
print(f"Best val F1: {best_val_f1:.4f}")
print(f"Total time: {total_time:.0f}s")
print(f"Best checkpoint: {best_ckpt_path}")

TRAINING: PVTv2B2_384_noaug_posweight
Config: 384x384, no augmentation, pos_weight=3.0
Encoder: tu-pvt_v2_b2
Parameters: 28,130,577
Trainable: 28,130,577
Size: 107.3 MB
Epoch   1/20 | TrLoss: 0.6126 TrAcc: 0.9013 | VlLoss: 0.4760 VlAcc: 0.9607 | ValF1: 0.5929 ValIoU: 0.4213 | LR: 1.00e-04 | 220.9s | Improved: YES | Patience: 0/7
Epoch   2/20 | TrLoss: 0.4032 TrAcc: 0.9742 | VlLoss: 0.4013 VlAcc: 0.9742 | ValF1: 0.7195 ValIoU: 0.5619 | LR: 1.00e-04 | 220.1s | Improved: YES | Patience: 0/7
Epoch   3/20 | TrLoss: 0.3279 TrAcc: 0.9821 | VlLoss: 0.3889 VlAcc: 0.9733 | ValF1: 0.7053 ValIoU: 0.5448 | LR: 1.00e-04 | 220.2s | Improved: NO | Patience: 1/7
Epoch   4/20 | TrLoss: 0.2851 TrAcc: 0.9861 | VlLoss: 0.4031 VlAcc: 0.9769 | ValF1: 0.7292 ValIoU: 0.5738 | LR: 1.00e-04 | 219.7s | Improved: YES | Patience: 0/7
Epoch   5/20 | TrLoss: 0.2581 TrAcc: 0.9887 | VlLoss: 0.3695 VlAcc: 0.9762 | ValF1: 0.7507 ValIoU: 0.6009 | LR: 1.00e-04 | 219.8s | Improved: YES | Patience: 0/7
Epoch   6/20 | TrLoss:

In [9]:
# Continue PVTv2 training from current model state up to 25 epochs

TARGET_EPOCHS = 25
current_epoch = len(history["train_loss"])
extra_epochs = TARGET_EPOCHS - current_epoch

print(f"Current completed epochs: {current_epoch}")
print(f"Continuing to: {TARGET_EPOCHS}")

if extra_epochs <= 0:
    print("Already reached target epoch count.")
else:
    # Keep existing best values if they exist
    best_val_iou = history.get("best_val_iou", best_val_iou)
    best_val_f1 = history.get("best_val_f1", best_val_f1)
    best_epoch = history.get("best_epoch", best_epoch)

    # If patience_counter exists from the previous loop, keep it.
    # Otherwise start from 0.
    try:
        patience_counter
    except NameError:
        patience_counter = 0

    for epoch in range(current_epoch + 1, TARGET_EPOCHS + 1):
        tl, ta, et = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            DEVICE
        )

        vl, va = validate(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        val_eval = full_evaluation(
            model,
            val_loader,
            DEVICE,
            threshold=BINARY_THRESHOLD
        )

        val_iou = val_eval["pixel"]["iou"]
        val_f1 = val_eval["pixel"]["f1"]
        val_precision = val_eval["pixel"]["precision"]
        val_recall = val_eval["pixel"]["recall"]

        scheduler.step(val_iou)

        current_lr = optimizer.param_groups[0]["lr"]
        total_time += et

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["train_acc"].append(ta)
        history["val_acc"].append(va)
        history["val_f1"].append(val_f1)
        history["val_iou"].append(val_iou)
        history["val_precision"].append(val_precision)
        history["val_recall"].append(val_recall)
        history["epoch_time"].append(et)
        history["lr"].append(current_lr)

        # Save latest checkpoint every epoch
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "history": history,
            "best_val_iou": best_val_iou,
            "best_val_f1": best_val_f1,
            "config": {
                "model_name": model_name,
                "encoder_name": encoder_name,
                "image_size": IMAGE_SIZE,
                "epochs": TARGET_EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "pos_weight": POS_WEIGHT,
                "augmentation": False,
                "binary_threshold": BINARY_THRESHOLD,
                "grad_clip_norm": GRAD_CLIP_NORM
            }
        }, latest_ckpt_path)

        if val_iou > best_val_iou:
            best_val_iou = val_iou
            best_val_f1 = val_f1
            best_epoch = epoch
            patience_counter = 0

            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "best_val_iou": best_val_iou,
                "best_val_f1": best_val_f1,
                "history": history,
                "total_params": total_params,
                "trainable_params": trainable_params,
                "model_size_mb": model_size,
                "config": {
                    "model_name": model_name,
                    "encoder_name": encoder_name,
                    "image_size": IMAGE_SIZE,
                    "epochs": TARGET_EPOCHS,
                    "batch_size": BATCH_SIZE,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "pos_weight": POS_WEIGHT,
                    "augmentation": False,
                    "binary_threshold": BINARY_THRESHOLD,
                    "grad_clip_norm": GRAD_CLIP_NORM
                }
            }, best_ckpt_path)

            improved = "YES"
        else:
            patience_counter += 1
            improved = "NO"

        print(f"Epoch {epoch:>3}/{TARGET_EPOCHS} | "
              f"TrLoss: {tl:.4f} TrAcc: {ta:.4f} | "
              f"VlLoss: {vl:.4f} VlAcc: {va:.4f} | "
              f"ValF1: {val_f1:.4f} ValIoU: {val_iou:.4f} | "
              f"LR: {current_lr:.2e} | "
              f"{et:.1f}s | Improved: {improved} | "
              f"Patience: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered after {PATIENCE} epochs without improvement.")
            break

    history["total_time"] = total_time
    history["best_epoch"] = best_epoch
    history["best_val_iou"] = best_val_iou
    history["best_val_f1"] = best_val_f1

    print("\nContinuation complete.")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val IoU: {best_val_iou:.4f}")
    print(f"Best val F1: {best_val_f1:.4f}")
    print(f"Best checkpoint: {best_ckpt_path}")

Current completed epochs: 19
Continuing to: 25
Epoch  20/25 | TrLoss: 0.0282 TrAcc: 0.9982 | VlLoss: 0.4762 VlAcc: 0.9824 | ValF1: 0.7791 ValIoU: 0.6381 | LR: 2.50e-05 | 218.3s | Improved: NO | Patience: 8/7

Early stopping triggered after 7 epochs without improvement.

Continuation complete.
Best epoch: 12
Best val IoU: 0.6861
Best val F1: 0.8138
Best checkpoint: /kaggle/working/best_PVTv2B2_384_noaug_posweight.pth


In [14]:
!ls -lh /kaggle/working
!find /kaggle/working -maxdepth 1 -type f -name "*PVTv2*" -print


total 215M
-rw-r--r-- 1 root root 108M Apr 28 16:46 best_PVTv2B2_384_noaug_posweight.pth
-rw-r--r-- 1 root root 108M Apr 28 17:23 latest_PVTv2B2_384_noaug_posweight.pth
drwxr-xr-x 5 root root 4.0K Apr 28 14:12 subset
/kaggle/working/best_PVTv2B2_384_noaug_posweight.pth
/kaggle/working/latest_PVTv2B2_384_noaug_posweight.pth


In [15]:
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])

print(f"Loaded best checkpoint from epoch {ckpt['epoch']}")
print(f"Best val IoU: {ckpt['best_val_iou']:.4f}")
print(f"Best val F1: {ckpt['best_val_f1']:.4f}")

results = full_evaluation(
    model,
    test_loader,
    DEVICE,
    threshold=BINARY_THRESHOLD
)

print(f"\n{'='*60}")
print(f"TEST SET — {model_name}")
print(f"{'='*60}")
print(f"Pixel Accuracy:  {results['pixel']['accuracy']:.4f}")
print(f"Pixel Precision: {results['pixel']['precision']:.4f}")
print(f"Pixel Recall:    {results['pixel']['recall']:.4f}")
print(f"Pixel F1:        {results['pixel']['f1']:.4f}")
print(f"IoU:             {results['pixel']['iou']:.4f}")
print(f"Dice:            {results['pixel']['dice']:.4f}")
print(f"Image F1:        {results['image']['f1']:.4f}")
print(f"{'='*60}")

Loaded best checkpoint from epoch 12
Best val IoU: 0.6861
Best val F1: 0.8138

TEST SET — PVTv2B2_384_noaug_posweight
Pixel Accuracy:  0.9833
Pixel Precision: 0.8359
Pixel Recall:    0.7522
Pixel F1:        0.7918
IoU:             0.6554
Dice:            0.7918
Image F1:        0.7476


In [17]:
save_data = {
    "model": model_name,
    "config": "384x384, no augmentation, pos_weight=3.0",
    "test_results": {
        "pixel_metrics": results["pixel"],
        "image_metrics": results["image"],
        "pixel_cm": results["pixel_cm"].tolist(),
        "image_cm": results["image_cm"].tolist(),
    },
    "history": history,
    "model_info": {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "model_size_mb": model_size,
    },
    "training_config": {
        "image_size": IMAGE_SIZE,
        "epochs": len(history["train_loss"]),
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "lr_patience": LR_PATIENCE,
        "lr_factor": LR_FACTOR,
        "lr_min": LR_MIN,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "binary_threshold": BINARY_THRESHOLD,
        "grad_clip_norm": GRAD_CLIP_NORM,
        "mixed_precision": MIXED_PRECISION,
        "seed": SEED,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
        "pos_weight": POS_WEIGHT,
        "augmentation": False
    },
    "best_checkpoint": {
        "best_epoch": ckpt["epoch"],
        "best_val_iou": ckpt["best_val_iou"],
        "best_val_f1": ckpt["best_val_f1"]
    }
}

json_path = f"{SAVE_DIR}/{model_name}_results.json"

with open(json_path, "w") as f:
    json.dump(save_data, f, indent=2, default=str)

print(f"✓ Saved JSON: {json_path}")

✓ Saved JSON: /kaggle/working/PVTv2B2_384_noaug_posweight_results.json
